# Raw ECG 환자 단위 SCD 학습·평가

`DL/data/final_split_v2_augmented.csv`의 PID·split·SCD를 그대로 사용합니다. 파일 형식은 현재 관측된 `(sample, 3)` float16을 기준으로 하며, 로더의 2 lead 처리는 합성 데이터로 확인했습니다. CNN → LSTM/Transformer → masked attention MIL → MLP 구조입니다.

전체 split 파일이 준비되면 같은 코드로 실행합니다. 원본 NPY에서 확인되지 않은 sample rate·배열 축는 upstream provenance와 대조해야 하며, `source_metadata_confirmed: true`는 확인 후에만 설정합니다. 30초 segment별 quality CSV(`pid,window_index,usable`)가 없으면 학습을 중단합니다. 구체적인 명령과 제한은 `DL/TRAINING.md`를 참고하세요.

유도 이름을 가정하지 않도록 공유 1유도 CNN을 모든 실제 유도에 적용한 뒤 평균 통합합니다. 배치의 유도 mask와 시간 mask는 별개이며, 양쪽 padding은 CNN/BatchNorm에서 제외합니다. 모델 선택 YAML은 `cnn_model`, `sequence_model`만 포함하고, threshold·MIL·MLP·학습 파라미터는 `training.yaml`에서 설정합니다.

환자 단위 MIL은 `training.yaml:mil.pooling`에서 `attention`(기본), `mean`, `max`를 선택합니다. Mean/max는 `mil: {pooling: mean}` 또는 `mil: {pooling: max}`처럼 설정하고 attention 전용 hidden_dim/activation/dropout을 제거합니다. 세 방식 모두 제외·패딩 시간 구간을 무시하고, 학습 snapshot으로 복원됩니다.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader

repo = Path.cwd().resolve()
while not (repo / "Members/sang/model/DL/model_choice.yaml").is_file():
    if repo.parent == repo:
        raise RuntimeError("저장소 내부에서 노트북을 실행하세요")
    repo = repo.parent
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

In [2]:
from Members.sang.model.config import (
    dl_config_paths,
    load_experiment_config,
    load_yaml,
    resolve_config_path,
)
from Members.sang.model.DL.composed import build_composed_model
from Members.sang.model.DL.dataloader.dataset import collate_patient_windows
from Members.sang.model.DL.dataloader.raw_waveform import RawPatientSequenceDataset, load_raw_cohort
from Members.sang.model.DL.dataloader.segmented_ecg import inspect_raw_cohort
from Members.sang.model.DL.raw_pipeline import train_raw_ecg
from Members.sang.model.inference import load_dl_run

choice_path = repo / "Members/sang/model/DL/model_choice.yaml"

/home/sang/sixletters-poc/Global/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 실행 설정·개인 계정

각 PC에서 한 번 `dev setup`으로 계정을 등록하면 이후 계정명을 노트북에 적지 않습니다. 환경 변수 `MLFLOW_TRACKING_*`와 `MLFLOW_EXPERIMENT_NAME`이 파일보다 우선합니다. 학습 전 표시되는 owner·username·experiment를 확인하세요. 기본은 고정 설정으로 학습 → validation → 최종 test → 개인 계정 MLflow 기록입니다. 모델 선택 중에는 test를 끕니다. 오프라인 실행은 `mlflow_logging=False`로 바꿉니다.

In [3]:
mlflow_logging = True
run_final_test = True
max_epochs = None
limit_train_patients = None
device_override = "auto"

if limit_train_patients is not None and run_final_test:
    raise ValueError("Smoke 학습에서는 run_final_test=False로 설정하세요.")

## 1. 설정과 현재 파일 감사

In [4]:
choice, basic, choice_path, basic_path = load_experiment_config(choice_path)
training = load_yaml(dl_config_paths(choice_path)["training"])
raw_config = basic["data"]["raw_ecg"]
records, inventory = inspect_raw_cohort(
    resolve_config_path(raw_config["split_path"], basic_path),
    resolve_config_path(raw_config["wave_directory"], basic_path),
)
display(pd.DataFrame(inventory["split_counts"]).T)
display(pd.DataFrame([record.__dict__ for record in records if record.reason]))
print("Split에 없는 파일:", inventory["wave_files_without_split_row"])
print("선택 모델:", choice["cnn_model"], choice["sequence_model"])

,patients,positive,negative,files_present,missing_or_invalid
train,349,96,253,349,[]
validation,71,8,63,71,[]
test,89,10,79,89,[]


""


Split에 없는 파일: ['P0001-1', 'P0001-2', 'P0002-1', 'P0002-2', 'P0003-1', 'P0003-2', 'P0004-1', 'P0004-2', 'P0006-1', 'P0006-2', 'P0007-1', 'P0007-2', 'P0009-1', 'P0009-2', 'P0010-1', 'P0010-2', 'P0013-1', 'P0013-2', 'P0014-1', 'P0014-2', 'P0015-1', 'P0015-2', 'P0017-1', 'P0017-2', 'P0018-1', 'P0018-2', 'P0019-1', 'P0019-2', 'P0022-1', 'P0022-2', 'P0023-1', 'P0023-2', 'P0024-1', 'P0024-2', 'P0029-1', 'P0029-2', 'P0033-1', 'P0033-2', 'P0036-1', 'P0036-2', 'P0037-1', 'P0037-2', 'P0041-1', 'P0041-2', 'P0042-1', 'P0042-2', 'P0043-1', 'P0043-2', 'P0045-1', 'P0045-2', 'P0048-1', 'P0048-2', 'P0049-1', 'P0049-2', 'P0050-1', 'P0050-2', 'P0051-1', 'P0051-2', 'P0052-1', 'P0052-2', 'P0053-1', 'P0053-2', 'P0055-1', 'P0055-2', 'P0059-1', 'P0059-2', 'P0061-1', 'P0061-2', 'P0062-1', 'P0062-2', 'P0063-1', 'P0063-2', 'P0064-1', 'P0064-2', 'P0068-1', 'P0068-2', 'P0071-1', 'P0071-2', 'P0072-1', 'P0072-2', 'P0076-1', 'P0076-2', 'P0079-1', 'P0079-2', 'P0082-1', 'P0082-2', 'P0086-1', 'P0086-2', 'P0088-1', 'P0088

## 2. 품질 인덱스 검증·로더·모델 생성

현재 quality CSV/provenance 확인이 없으면 이 셀에서 명확한 오류로 중단합니다. 누락 파일 때문에 환자를 다른 split으로 옮기지 않습니다. 모델 입력은 최대 `training.yaml:max_segments_per_patient`개의 유효 segment이며, 균등 간격으로 선택한 원래 segment 위치를 보존합니다. 정규화 기본값은 `none`입니다.

In [5]:
cohorts, usable_indices, audit = load_raw_cohort(basic, basic_path, require_quality=True)
display(pd.DataFrame(audit["split_summary"]).T)
max_segments = int(training["max_segments_per_patient"])
train_dataset = RawPatientSequenceDataset(
    cohorts["train"], usable_indices, raw_config, max_segments, raw_config["normalization"]
)
train_loader = DataLoader(
    train_dataset,
    batch_size=int(training["batch_size"]),
    shuffle=False,
    collate_fn=collate_patient_windows,
)
model = build_composed_model(choice, basic, choice_path=choice_path)
print("Train 환자:", len(train_dataset))
print("학습 parameter 수:", sum(p.numel() for p in model.parameters() if p.requires_grad))
batch = next(iter(train_loader))
print("첫 배치 PID:", batch[0], "waveform shape:", tuple(batch[1].shape))
del batch, model, train_loader, train_dataset

,patients,positive,negative
train,349,96,253
validation,71,8,63
test,89,10,79


Train 환자: 349
학습 parameter 수: 120674
첫 배치 PID: ['P0001'] waveform shape: (1, 128, 3, 6000)


## 3. Train·validation·best checkpoint와 MLflow

기본은 전체 학습입니다. `mlflow_logging=True`이면 로컬 `.poc.local.toml` 및 MLflow 환경 변수의 개인 계정을 확인한 뒤 학습하고, 완료한 checkpoint·architecture·집계 validation 지표를 자동 기록합니다. 연결 확인이 실패하면 학습 전에 중단합니다. 학습 도중 전송이 실패하면 로컬 모델을 보존하고 `PENDING`으로 기록합니다.

짧은 smoke는 위 실행 설정에서 `max_epochs=1`, `limit_train_patients=16`, `device_override="cpu"`, `run_final_test=False`로 바꿉니다. Train만 가중치를 학습하고 validation AP로 checkpoint를 선택합니다.

In [ ]:
run_dir = train_raw_ecg(
    choice_path,
    max_epochs=max_epochs,
    limit_train_patients=limit_train_patients,
    device_override=device_override,
    mlflow_logging=mlflow_logging,
)
print(run_dir)
display(pd.read_csv(run_dir / "training_history.csv"))
if mlflow_logging:
    print(json.loads((run_dir / "mlflow_status.json").read_text()))

## 4. Checkpoint 복원·validation 결과 확인

In [ ]:
restored_model, metadata, resolved, device = load_dl_run(run_dir, torch.device("cpu"))
print("복원 모델:", metadata["model"], "best epoch:", metadata["best_epoch"])
print(json.loads((run_dir / "metrics.json").read_text())["validation"])
display(pd.read_csv(run_dir / "validation_predictions.csv").head())
del restored_model

## 5. 최종 test 평가

위 실행 설정의 `run_final_test=True`이면 학습 후 고정된 best checkpoint·threshold로 test 평가하고, `mlflow_logging=True`이면 같은 개인 계정으로 평가 결과를 기록합니다. 설정을 비교·조정하는 중에는 `run_final_test=False`를 사용합니다. Test 평가가 기반 모델을 재학습하지는 않습니다. 원본 `eval.yaml`은 수정하지 않습니다.

In [ ]:
if run_final_test:
    from tempfile import TemporaryDirectory

    import yaml
    from Members.sang.model.evaluation.evaluation import evaluate_saved_model

    eval_path = repo / "Members/sang/model/evaluation/eval.yaml"
    eval_config = load_yaml(eval_path)
    eval_config["basic_config"] = str(basic_path)
    eval_config["evaluation"].update(
        target="dl",
        model_path=str(run_dir / "best_model.pt"),
        artifact_dir=None,
        split="test",
        mlflow_logging=mlflow_logging,
        threshold_override=None,
        results_directory=str(
            resolve_config_path(eval_config["evaluation"]["results_directory"], eval_path)
        ),
    )
    with TemporaryDirectory(prefix="raw-dl-evaluation-") as temporary:
        temporary_config = Path(temporary) / "eval.yaml"
        temporary_config.write_text(yaml.safe_dump(eval_config), encoding="utf-8")
        evaluation_dir = evaluate_saved_model(temporary_config)
    print(evaluation_dir)
    display(pd.read_csv(evaluation_dir / "predictions.csv").head())

## 기존 SQI-window 경로

기존 10초 window source는 별도 경로입니다. 필요한 경우 저장소 루트에서 `Global/.venv/bin/python -m Members.sang.model.DL.training`으로 실행합니다. Raw 학습에 기존 `sqi_pass` sidecar를 재사용하지 않습니다.

## 전송 실패 시 재시도

학습은 `from Members.sang.model.tracking.log_dl import log_dl_training_to_mlflow; log_dl_training_to_mlflow(run_dir)`로, 평가는 `from Members.sang.model.tracking.log_evaluation import log_evaluation_to_mlflow; log_evaluation_to_mlflow(evaluation_dir)`로 재학습·재평가 없이 전송합니다. 로컬 `mlflow_status.json`에서 `FINISHED` 또는 `PENDING`을 확인합니다. 환자별 예측·품질 인덱스·PID audit은 자동 업로드하지 않습니다.